In [1]:
import requests
from dotenv import load_dotenv
import os
import json
import time
from typing import Dict

load_dotenv()

api_key = os.getenv("RAPID_API_KEY")

In [3]:
with open("manhattan_listings.json", "r", encoding="utf-8") as f:
    listings = json.load(f)

ids = [listing["id"] for listing in listings if "id" in listing]

print(f"Extracted {len(ids)} listing IDs")

# check if ids are unique
if len(ids) == len(set(ids)):
    print("Ids are all unique")
else:
    print("IDS ARE NOT UNIQUE. PLEASE CORRECT BEFORE GETTING APARTMENT DETAILS")

Extracted 6432 listing IDs
Ids are all unique


In [4]:
def load_existing_details() -> Dict:
    """Load existing apartment details."""
    if os.path.exists("manhattan_details.json"):
        with open("manhattan_details.json", "r", encoding="utf-8") as f:
            details = json.load(f)
            return {str(item["id"]): item for item in details}
    return {}

# Replace the second cell with:
# Load changed listings
with open("changed_listings.json", "r", encoding="utf-8") as f:
    changed_listings = json.load(f)

# Load current listings to know what should be deleted
with open("manhattan_listings.json", "r", encoding="utf-8") as f:
    current_listings = json.load(f)
    current_ids = set(str(listing["id"]) for listing in current_listings)

# Load existing details
existing_details = load_existing_details()

# Identify listings to process
new_ids = [str(listing["id"]) for listing in changed_listings]
print(f"Found {len(new_ids)} listings to update")

# Identify listings to remove
removed_ids = set(existing_details.keys()) - current_ids
if removed_ids:
    print(f"Found {len(removed_ids)} listings to remove")

Found 1552 listings to update
Found 1559 listings to remove


In [5]:
headers = {
    "x-rapidapi-key": api_key,
    "x-rapidapi-host": "streeteasy-api.p.rapidapi.com"
}

# Fetch details for new/updated listings
for i, listing_id in enumerate(new_ids, 1):
    url = f"https://streeteasy-api.p.rapidapi.com/rentals/{listing_id}"
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        existing_details[listing_id] = r.json()
        print(f"Fetched {i}/{len(new_ids)}: {listing_id}")
    except requests.RequestException as e:
        print(f"Error fetching {listing_id}: {e}")
        continue
    time.sleep(0.2)

# Remove listings that no longer exist
for removed_id in removed_ids:
    existing_details.pop(removed_id, None)

# Save updated details
with open("manhattan_details.json", "w", encoding="utf-8") as f:
    json.dump(list(existing_details.values()), f, indent=2)

print(f"Updated details: {len(new_ids)} new/changed, {len(removed_ids)} removed")

Fetched 1/1552: 4894590
Fetched 2/1552: 4894589
Fetched 3/1552: 4894580
Fetched 4/1552: 4894578
Fetched 5/1552: 4894577
Fetched 6/1552: 4894576
Fetched 7/1552: 4894575
Fetched 8/1552: 4894574
Fetched 9/1552: 4894573
Fetched 10/1552: 4894572
Fetched 11/1552: 4894570
Fetched 12/1552: 4894547
Fetched 13/1552: 4894544
Fetched 14/1552: 4894543
Fetched 15/1552: 4894541
Fetched 16/1552: 4894540
Fetched 17/1552: 4894538
Fetched 18/1552: 4894537
Error fetching 4894507: 502 Server Error: Bad Gateway for url: https://streeteasy-api.p.rapidapi.com/rentals/4894507
Fetched 20/1552: 4894500
Fetched 21/1552: 4894499
Fetched 22/1552: 4894465
Fetched 23/1552: 4894462
Fetched 24/1552: 4894460
Fetched 25/1552: 4894459
Fetched 26/1552: 4894412
Fetched 27/1552: 4894411
Fetched 28/1552: 4894410
Error fetching 4894409: 504 Server Error: Gateway Time-out for url: https://streeteasy-api.p.rapidapi.com/rentals/4894409
Fetched 30/1552: 4894375
Fetched 31/1552: 4894370
Fetched 32/1552: 4894369
Fetched 33/1552: 489

In [5]:
# converts json to csv, no longer needed


# with open("manhattan_details.json", "r") as f:
#     data = json.load(f)

# df = pd.json_normalize(
#     data,
#     sep="_",  # replaces nested keys with underscore, e.g. building_id
# )

# # Convert list-type columns to comma-separated strings
# for col in df.columns:
#     df[col] = df[col].apply(
#         lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
#     )

# # Save to CSV
# df.to_csv("manhattan_details.csv", index=False)
# print("json to csv conversion successful")